# 03. Datenvorbereitung & Feature Engineering (Preprocessing)

## **Ziel des Notebooks**
In diesem Notebook wird die transaktionale Rohdatengrundlage (`traveltide_raw_data.csv`) auf Nutzerebene aggregiert. Das Ziel ist es, aus den Verlaufs- und Sitzungsdaten aussagekräftige Kundenmerkmale (*Features*) zu berechnen, um eine konsolidierte Matrix (1 Zeile pro `user_id`) für das anschließende Machine-Learning-Clustering zu erstellen.

---

## **Kernaktivitäten & Durchführung**

1. **Laden & Bereinigen der Daten:**
   * Import der gefilterten Daten aus `data/traveltide_raw_data.csv`.
   * Prüfung und Umwandlung von Datentypen (z. B. Zeitstempel, Datumsfelder).
   * Behandlung von Fehlwerten (*Null Values*) und Ausreißern.

2. **Feature Engineering (Aggregation auf Nutzerebene):**
   * **Aktivität & Engagement:** Berechnung von Gesamtsitzungen, durchschnittlicher Sitzungsdauer und Klickzahlen.
   * **Buchung Verhalten:** Ermittlung von Flug- und Hotelbuchungsraten, Stornierungsquoten sowie bevorzugten Reisezeiten.
   * **Finanzielle Metriken:** Berechnung der Gesamtausgaben, der durchschnittlichen Ausgaben pro Reise sowie des Rabattnutzungsverhaltens (*Discount Sensitivity*).

3. **Demografische Einbindung & Encoding:**
   * Integration von Nutzerattributen (Alter, Geschlecht, Familienstand, Kinder).
   * Transformation kategorialer Variablen in numerische Formats (z. B. One-Hot-Encoding).

4. **Skalierung & Datensatz-Export:**
   * Normalisierung/Standardisierung der Features für abstandsbasierte ML-Algorithmen (z. B. K-Means).
   * Export des finalen Preprocessing-Datensatzes (`traveltide_features.csv`).

---

## **Erwartetes Ergebnis**
Eine bereinigte, skalierten Datenmatrix, in der jeder aktive Nutzer durch ein eindeutiges Verhaltensprofil repräsentiert wird – bereit für das Kunden-Clustering in Phase 4.

In [1]:
import pandas as pd
import numpy as np

# 1. Gefilterte Rohdaten aus Phase 1 laden
df_raw = pd.read_csv('../data/traveltide_raw_data.csv')

print(f"Geladene Datensätze: {len(df_raw):,}")
print("Spalten im Datensatz:", df_raw.columns.tolist())

# 2. Vorschau auf die Rohdaten
df_raw.head()

Geladene Datensätze: 5,998
Spalten im Datensatz: ['user_id', 'birthdate', 'gender', 'married', 'has_children', 'home_airport', 'total_sessions', 'total_page_clicks', 'avg_session_duration_sec', 'avg_flight_discount', 'avg_hotel_discount', 'total_cancellations', 'total_flights_booked', 'total_checked_bags', 'total_hotels_booked', 'avg_hotel_rooms', 'avg_hotel_nights']


,user_id,birthdate,gender,married,has_children,home_airport,total_sessions,total_page_clicks,avg_session_duration_sec,avg_flight_discount,avg_hotel_discount,total_cancellations,total_flights_booked,total_checked_bags,total_hotels_booked,avg_hotel_rooms,avg_hotel_nights
0,23557,1958-12-08,F,True,False,LGA,8,82,76.625000,0.000,0.175,0,0,0,2,1.5,10.0
1,94883,1972-03-16,F,True,False,MCI,8,73,67.750000,0.000,0.100,0,2,1,2,1.5,0.5
2,101486,1972-12-07,F,True,True,TCM,8,131,122.250000,0.075,0.000,0,1,0,2,1.5,4.0
3,101961,1980-09-14,F,True,False,BOS,8,126,117.750000,0.150,0.100,0,5,2,5,1.0,3.8
4,106907,1978-11-17,F,True,True,TNT,8,240,758.915066,0.000,0.000,1,1,10,1,3.0,11.0


In [2]:
# 1. Alter berechnen (Stichtag 2023)
df_features = df_raw.copy()
df_features['birthdate'] = pd.to_datetime(df_features['birthdate'])
df_features['age'] = 2023 - df_features['birthdate'].dt.year

# 2. Bools in Integer umwandeln (True/False -> 1/0)
df_features['married'] = df_features['married'].astype(int)
df_features['has_children'] = df_features['has_children'].astype(int)

# 3. Geschlecht binär kodieren (F=1, M=0)
df_features['is_female'] = (df_features['gender'] == 'F').astype(int)

# 4. Fehlwerte bei Discounts/Metriken mit 0 auffüllen (kein Rabatt genutzt)
df_features = df_features.fillna(0)

# Überprüfen der aufbereiteten Features
print("Form des aufbereiteten Datensatzes:", df_features.shape)
df_features[['user_id', 'age', 'married', 'has_children', 'is_female']].head()

Form des aufbereiteten Datensatzes: (5998, 19)


,user_id,age,married,has_children,is_female
0,23557,65,1,0,1
1,94883,51,1,0,1
2,101486,51,1,1,1
3,101961,43,1,0,1
4,106907,45,1,1,1


In [4]:
from sklearn.preprocessing import StandardScaler

# 1. Relevante Features für das Clustering auswählen (ohne IDs/Texte)
feature_cols = [col for col in df_features.columns if col not in ['user_id', 'birthdate', 'gender', 'home_airport']]

# 2. Skalierung mit StandardScaler (Z-Score Normalisierung)
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_features[feature_cols]),
    columns=feature_cols
)

# 3. user_id wieder zur Identifikation anfügen
df_scaled.insert(0, 'user_id', df_features['user_id'].values)

# 4. Daten speichern für Phase 4 (Clustering)
df_features.to_csv('../data/traveltide_features_unscaled.csv', index=False)
df_scaled.to_csv('../data/traveltide_features_scaled.csv', index=False)

print("--- PREPROCESSING ERFOLGREICH ABGESCHLOSSEN ---")
print(f"Verarbeitete Nutzer: {len(df_scaled):,}")
print("Gespeicherte Dateien: 'traveltide_features_unscaled.csv' und 'traveltide_features_scaled.csv'")

--- PREPROCESSING ERFOLGREICH ABGESCHLOSSEN ---
Verarbeitete Nutzer: 5,998
Gespeicherte Dateien: 'traveltide_features_unscaled.csv' und 'traveltide_features_scaled.csv'


### **Preprocessing & Skalierung – Kernerkenntnisse**

* **Datenbereinigung:** Fehlwerte in verhaltensbezogenen Metriken wurden mit `0` aufgefüllt (z. B. Nutzer ohne bisherige Rabattnutzung).
* **Feature Transformation:** Kategoriale & zeitliche Variablen wurden in numerische Signale umgewandelt (`age` berechnet, `married`, `has_children` sowie `is_female` binär kodiert).
* **Normalisierung:** Alle kontinuierlichen Features wurden mittels `StandardScaler` (Z-Score) standardisiert, um Verzerrungen im euklidischen Abstand des K-Means-Algorithmus zu verhindern.
* **Outputs:** 
  * Unskalierter Datensatz: `traveltide_features_unscaled.csv` (für die spätere Interpretation der Cluster-Mittelwerte in echten Werten).
  * Skalierter Datensatz: `traveltide_features_scaled.csv` (direkter Input für Phase 4: Clustering).